# Ablation RV-A — initial condition only

The reversed single vortex, T = 2:

    u = -sin^2(pi x) sin(2 pi y) cos(pi t / T)
    v =  sin^2(pi y) sin(2 pi x) cos(pi t / T)

The flow reverses at t = T/2, so the interface stretches into a thin filament
and then returns to its initial shape; the final error is magnified by the
reversal rather than cancelling with it. Unlike rotation, the exact solution is
NOT a signed distance function for t > 0 -- 87% of the domain violates
|grad phi| = 1, against 0.3% under rotation.

Velocity fixed; only the initial interface varies. This is the standard
single-input-function protocol in the operator literature, so the numbers are
comparable to it.

Three arms x three seeds, 16 training instances.

| | |
|---|---|
| grid | 64 x 64 x 32 |
| test set | 100 held-out instances |
| Adam | 20,000 steps, batch 4 |
| model | width 20, modes (12,12,8) |
| eikonal weight | SAW, zero-seeded |
| seeds | 42, 43, 44 |

No L-BFGS stage: Adam at 20k reaches the same place on these benchmarks, and
adding a refinement that helps one arm more than another would confound the
comparison.

The eikonal weight uses `--saw_init zero`. Under a hard initial condition the
field starts exactly at phi_0, which IS a signed distance function, so g_eik
starts near zero, the raw ratio saturates, and the published seeding pins the
weight at its clamp for thousands of steps. Seeding at zero lets it climb as
the field genuinely departs from phi_0.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/NORV
!ls *.py

/content/drive/MyDrive/NORV
collect_results.py  fno3d.py	  rv_family.py	     visualize.py
eval_checkpoint.py  residuals.py  train_operator.py


### Characteristics

No closed form here. References come from backward characteristics -- advecting
each evaluation point back to t = 0 along the flow and evaluating phi_0 there,
RK45 at rtol 1e-8. Cached to disk; runs only the first time.

In [3]:
import rv_family
x0, y0 = rv_family._characteristics(64, 64, 32)
print('cached:', x0.shape)

cached: (64, 64, 32)


### Sanity check

The vortex is divergence-free, so the enclosed area must match at t = 0 and
t = T; any difference is discretisation and sets the floor below which a mass
number cannot be read.

In [4]:
import torch, rv_family as rv
from residuals import eikonal_pointwise
X, Y, Tt, h = rv.make_grid(64, 64, 32)
u, v = rv.velocity(X, Y, Tt)
div = (u[2:,1:-1] - u[:-2,1:-1]) / (2*h[0]) + (v[1:-1,2:] - v[1:-1,:-2]) / (2*h[1])
print(f'max |div u|                {float(div.abs().max()):.2e}')
p = rv.sample_family(4, seed=42)
phi0, ex, _ = rv.build_dataset(p, 64, 64, 32)
print('area t=0                  ', (ex[:, ..., 0] < 0).float().sum(dim=(1,2)).tolist())
print('area t=T                  ', (ex[:, ..., -1] < 0).float().sum(dim=(1,2)).tolist())
print(f'drift of exact             {float(rv.mass_drift(ex, h[0], h[1]).mean()):.3f}%  <- floor')
r = eikonal_pointwise(ex, h)
print(f'eikonal resid. of exact    median {float(r.median()):.2e}, '
      f'{float((r > 1e-3).float().mean()):.1%} of points violate |grad phi|=1')

max |div u|                1.41e-05
area t=0                   [220.0, 171.0, 356.0, 305.0]
area t=T                   [220.0, 171.0, 356.0, 305.0]
drift of exact             1.089%  <- floor
eikonal resid. of exact    median 8.81e-02, 86.9% of points violate |grad phi|=1


---
## Training

### data-free (physics only)

In [5]:
!python train_operator.py --benchmark rv --loss strong --saw --steps 20000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RV  (velocity fixed)
SAW: q=0.95, beta=0.999, init=ratio
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 1.793e-02 | train  25.847% | test  25.882% | mass  76.77% | drift  79.26% | w_eik 9.081e-01 | 0.6m
   500 loss 1.142e-02 | train  23.706% | test  23.729% | mass  81.99% | drift  84.65% | w_eik 8.199e-01 | 1.1m
   750 loss 9.336e-03 | train  22.817% | test  22.836% | mass  84.01% | drift  86.73% | w_eik 7.457e-01 | 1.7m
  1000 loss 7.707e-03 | train  22.700% | test  22.719% | mass  88.81% | drift  91.68% | w_eik 7.300e-01 | 2.2m
  1250 loss 6.888e-03 | train  23.094% | test  23.112% | mass  88.76% | drift  91.63% | w_eik 6.998e-01 | 2.8m
  1500 loss 6.629e-03 | train  23.264% | test  23.283% | mass  88.40% | drift  91.26% | w_eik 6.746e-01 | 3.3m
  1750 loss 5.800e-03 | train  23.462% | test  23.480% | mass  88.92% | drift  91.79% | w_eik 6.255e-01 | 3.9m
  2000 los

In [6]:
!python train_operator.py --benchmark rv --loss strong --saw --soft_ic --w_ic 10 --steps 20000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RV  (velocity fixed)
SAW: q=0.95, beta=0.999, init=ratio
labels: 0/16 instances (data-free)  lambda_data=1
IC: soft (w_ic=10)
params=7.38M  train=16  test=100
   250 loss 3.699e-02 | train  26.695% | test  27.067% | mass  88.98% | drift  88.27% | ic 1.10e-03 | w_eik 9.991e-01 | 0.6m
   500 loss 1.782e-02 | train  25.323% | test  25.481% | mass  84.62% | drift  84.21% | ic 3.57e-04 | w_eik 9.989e-01 | 1.1m
   750 loss 1.351e-02 | train  25.069% | test  25.152% | mass  82.45% | drift  82.39% | ic 9.56e-05 | w_eik 9.990e-01 | 1.7m
  1000 loss 1.173e-02 | train  24.810% | test  24.885% | mass  82.62% | drift  83.78% | ic 5.39e-05 | w_eik 9.989e-01 | 2.2m
  1250 loss 1.071e-02 | train  24.955% | test  25.021% | mass  82.20% | drift  83.70% | ic 4.14e-05 | w_eik 9.985e-01 | 2.8m
  1500 loss 9.802e-03 | train  24.819% | test  24.886% | mass  85.46% | drift  86.97% | ic 3.99e-05 | w_eik 9.975e-01 | 3.4m
  1750 loss 8.968e-03 | train  24.990% |

In [7]:
!python train_operator.py --benchmark rv --loss strong --w_eik 0 --steps 20000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RV  (velocity fixed)
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 2.843e-03 | train   8.541% | test   8.555% | mass  48.84% | drift  50.39% | 0.5m
   500 loss 1.072e-03 | train   6.773% | test   6.721% | mass  31.19% | drift  32.15% | 1.1m
   750 loss 5.512e-04 | train   5.411% | test   5.480% | mass  27.86% | drift  28.67% | 1.6m
  1000 loss 3.956e-04 | train   4.267% | test   4.285% | mass  25.55% | drift  26.32% | 2.1m
  1250 loss 2.654e-04 | train   3.347% | test   3.316% | mass  19.35% | drift  19.92% | 2.7m
  1500 loss 1.468e-04 | train   2.585% | test   2.596% | mass  10.97% | drift  11.29% | 3.2m
  1750 loss 1.083e-04 | train   2.293% | test   2.304% | mass   8.83% | drift   9.15% | 3.7m
  2000 loss 1.083e-04 | train   2.044% | test   2.082% | mass   7.32% | drift   7.57% | 4.2m
  2250 loss 1.095e-04 | train   1.931% | test   1.983% | mass   7.06% | drift  

In [ ]:
!python train_operator.py --benchmark rv --loss strong --saw --saw_init zero --steps 20000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RV  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 4.652e-03 | train  10.894% | test  10.940% | mass  64.78% | drift  66.89% | w_eik 1.543e-02 | 0.6m
   500 loss 2.533e-03 | train   9.485% | test   9.512% | mass  55.61% | drift  57.43% | w_eik 1.358e-02 | 1.1m
   750 loss 1.832e-03 | train   8.523% | test   8.573% | mass  51.02% | drift  52.71% | w_eik 1.148e-02 | 1.7m
  1000 loss 1.475e-03 | train   7.681% | test   7.714% | mass  50.70% | drift  52.37% | w_eik 9.662e-03 | 2.2m
  1250 loss 1.238e-03 | train   6.967% | test   6.979% | mass  47.95% | drift  49.52% | w_eik 8.097e-03 | 2.8m
  1500 loss 1.035e-03 | train   6.317% | test   6.329% | mass  43.67% | drift  45.10% | w_eik 6.755e-03 | 3.3m
  1750 loss 9.008e-04 | train   5.874% | test   5.883% | mass  44.03% | drift  45.48% | w_eik 5.644e-03 | 3.9m
  2000 loss

In [ ]:
!python train_operator.py --benchmark rv --loss strong --saw --saw_init zero --steps 20000 --n_train 16 --n_test 100 --seed 43

device=cuda  grid=64x64x32  loss=strong
benchmark: RV  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 3.813e-03 | train  10.623% | test  10.564% | mass  60.70% | drift  62.62% | w_eik 1.525e-02 | 0.5m
   500 loss 2.738e-03 | train   9.412% | test   9.587% | mass  54.79% | drift  56.51% | w_eik 1.337e-02 | 1.1m
   750 loss 1.907e-03 | train   8.656% | test   8.941% | mass  51.83% | drift  53.44% | w_eik 1.127e-02 | 1.7m
  1000 loss 1.506e-03 | train   7.898% | test   8.256% | mass  50.00% | drift  51.55% | w_eik 9.441e-03 | 2.2m
  1250 loss 1.416e-03 | train   7.097% | test   7.452% | mass  48.40% | drift  49.87% | w_eik 7.953e-03 | 2.8m
  1500 loss 1.045e-03 | train   6.636% | test   6.941% | mass  48.34% | drift  49.79% | w_eik 6.792e-03 | 3.3m
  1750 loss 8.876e-04 | train   6.248% | test   6.490% | mass  47.54% | drift  48.97% | w_eik 5.846e-03 | 3.9m
  2000 loss

In [ ]:
!python train_operator.py --benchmark rv --loss strong --saw --saw_init zero --steps 20000 --n_train 16 --n_test 100 --seed 44

device=cuda  grid=64x64x32  loss=strong
benchmark: RV  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 3.858e-03 | train  10.972% | test  10.851% | mass  59.39% | drift  61.32% | w_eik 1.607e-02 | 0.5m
   500 loss 2.474e-03 | train   9.852% | test   9.732% | mass  57.95% | drift  59.83% | w_eik 1.437e-02 | 1.1m
   750 loss 2.030e-03 | train   9.019% | test   8.920% | mass  53.39% | drift  55.11% | w_eik 1.236e-02 | 1.7m
  1000 loss 1.870e-03 | train   8.261% | test   8.180% | mass  52.32% | drift  54.01% | w_eik 1.048e-02 | 2.2m
  1250 loss 1.635e-03 | train   7.527% | test   7.487% | mass  52.45% | drift  54.14% | w_eik 8.920e-03 | 2.8m
  1500 loss 1.361e-03 | train   7.033% | test   7.028% | mass  52.61% | drift  54.32% | w_eik 7.567e-03 | 3.3m
  1750 loss 9.764e-04 | train   6.507% | test   6.521% | mass  50.01% | drift  51.63% | w_eik 6.410e-03 | 3.9m
  2000 loss

In [ ]:
!python collect_results.py

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
data-free   fixed   16    64^2x32  20000 42   1.503%   1.543%    2.67%   45.0
data-free   fixed   16    64^2x32  20000 43   1.595%   1.624%    3.94%   44.9
data-free   fixed   16    64^2x32  20000 44   1.658%   1.676%    4.11%   44.9

arm         v        n  steps seeds  test mean     std  mass mean     std
-------------------------------------------------------------------------
data-free   fixed   16  20000     3     1.614%  0.067%      3.57%   0.79%


### supervised (labels only)

In [ ]:
!python train_operator.py --benchmark rv --loss supervised --steps 20000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=supervised
benchmark: RV  (velocity fixed)
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 7.761e-04 | train   6.613% | test   6.666% | mass  25.85% | drift  26.61% | 0.5m
   500 loss 2.050e-04 | train   3.444% | test   3.439% | mass  13.79% | drift  14.17% | 1.0m
   750 loss 5.276e-05 | train   2.205% | test   2.338% | mass  10.97% | drift  11.28% | 1.6m
  1000 loss 3.368e-05 | train   1.722% | test   1.914% | mass   9.00% | drift   9.29% | 2.1m
  1250 loss 2.888e-05 | train   1.453% | test   1.626% | mass   6.83% | drift   7.07% | 2.6m
  1500 loss 2.111e-05 | train   1.249% | test   1.437% | mass   5.27% | drift   5.49% | 3.2m
  1750 loss 1.763e-05 | train   1.083% | test   1.281% | mass   4.80% | drift   5.02% | 3.7m
  2000 loss 1.287e-05 | train   0.942% | test   1.121% | mass   4.81% | drift   4.97% | 4.2m
  2250 loss 1.103e-05 | train   0.877% | test   1.059% | mass   4.89% | drift   5.07% | 4.8m
  2500 loss 7.363e-06 | train   

In [ ]:
!python train_operator.py --benchmark rv --loss supervised --steps 20000 --n_train 16 --n_test 100 --seed 43

device=cuda  grid=64x64x32  loss=supervised
benchmark: RV  (velocity fixed)
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 4.775e-04 | train   5.711% | test   5.960% | mass  31.94% | drift  33.01% | 0.5m
   500 loss 1.478e-04 | train   3.380% | test   3.760% | mass  19.09% | drift  19.74% | 1.1m
   750 loss 1.247e-04 | train   2.445% | test   2.819% | mass  13.82% | drift  14.36% | 1.6m
  1000 loss 4.228e-05 | train   1.878% | test   2.221% | mass   9.59% | drift   9.95% | 2.2m
  1250 loss 3.629e-05 | train   1.405% | test   1.776% | mass   8.80% | drift   9.17% | 2.7m
  1500 loss 2.027e-05 | train   1.284% | test   1.639% | mass   9.41% | drift   9.91% | 3.3m
  1750 loss 1.642e-05 | train   1.180% | test   1.514% | mass   6.93% | drift   7.16% | 3.8m
  2000 loss 1.571e-05 | train   1.023% | test   1.362% | mass   6.61% | drift   6.98% | 4.4m
  2250 loss 1.066e-05 | train   0.937% | test   1.303% | mass   6.45% | drift   6.88% | 4.9m
  2500 loss 9.962e-06 | train   

In [ ]:
!python train_operator.py --benchmark rv --loss supervised --steps 20000 --n_train 16 --n_test 100 --seed 44

device=cuda  grid=64x64x32  loss=supervised
benchmark: RV  (velocity fixed)
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 5.390e-04 | train   6.152% | test   6.180% | mass  29.07% | drift  30.02% | 0.6m
   500 loss 2.306e-04 | train   3.902% | test   3.971% | mass  17.75% | drift  18.35% | 1.1m
   750 loss 8.530e-05 | train   2.738% | test   2.835% | mass  13.72% | drift  14.09% | 1.7m
  1000 loss 8.466e-05 | train   2.249% | test   2.216% | mass   7.69% | drift   7.92% | 2.2m
  1250 loss 7.244e-05 | train   1.610% | test   1.660% | mass   6.27% | drift   6.40% | 2.8m
  1500 loss 3.735e-05 | train   1.410% | test   1.439% | mass   5.14% | drift   5.35% | 3.3m
  1750 loss 1.493e-05 | train   1.141% | test   1.189% | mass   5.18% | drift   5.26% | 3.9m
  2000 loss 1.993e-05 | train   1.096% | test   1.121% | mass   3.81% | drift   3.99% | 4.4m
  2250 loss 1.409e-05 | train   0.940% | test   0.977% | mass   3.11% | drift   3.18% | 5.0m
  2500 loss 8.379e-06 | train   

In [ ]:
!python collect_results.py

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
data-free   fixed   16    64^2x32  20000 42   1.503%   1.543%    2.67%   45.0
data-free   fixed   16    64^2x32  20000 43   1.595%   1.624%    3.94%   44.9
data-free   fixed   16    64^2x32  20000 44   1.658%   1.676%    4.11%   44.9
supervised  fixed   16    64^2x32  20000 42   0.256%   0.368%    1.14%   42.9
supervised  fixed   16    64^2x32  20000 43   0.262%   0.405%    1.22%   43.9
supervised  fixed   16    64^2x32  20000 44   0.284%   0.335%    1.03%   44.0

arm         v        n  steps seeds  test mean     std  mass mean     std
-------------------------------------------------------------------------
data-free   fixed   16  20000     3     1.614%  0.067%      3.57%   0.79%
supervised  fixed   16  20000     3     0.369%  0.035%      1.13%   0.10%


### hybrid (8 of 16 labelled)

In [ ]:
!python train_operator.py --benchmark rv --loss strong --saw --saw_init zero --n_labelled 8 --steps 20000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RV  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 8/16 instances  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 5.893e-03 | train   8.722% | test   8.736% | mass  50.44% | drift  52.09% | data 1.19e-03 | w_eik 1.659e-02 | 0.6m
   500 loss 3.320e-03 | train   7.405% | test   7.407% | mass  44.66% | drift  46.13% | data 6.54e-04 | w_eik 1.468e-02 | 1.1m
   750 loss 2.475e-03 | train   6.150% | test   6.165% | mass  38.38% | drift  39.65% | data 4.89e-04 | w_eik 1.255e-02 | 1.7m
  1000 loss 2.013e-03 | train   5.283% | test   5.310% | mass  31.54% | drift  32.59% | data 4.19e-04 | w_eik 1.068e-02 | 2.3m
  1250 loss 1.730e-03 | train   4.574% | test   4.596% | mass  30.16% | drift  31.16% | data 3.48e-04 | w_eik 9.062e-03 | 2.8m
  1500 loss 1.468e-03 | train   4.178% | test   4.211% | mass  25.32% | drift  26.16% | data 2.39e-04 | w_eik 7.676e-03 | 3.4m
  1750 loss 1.264e-03 | train   3.784%

In [ ]:
!python train_operator.py --benchmark rv --loss strong --saw --saw_init zero --n_labelled 8 --steps 20000 --n_train 16 --n_test 100 --seed 43

device=cuda  grid=64x64x32  loss=strong
benchmark: RV  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 8/16 instances  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 5.143e-03 | train   8.807% | test   8.774% | mass  52.87% | drift  54.54% | data 1.15e-03 | w_eik 1.649e-02 | 0.6m
   500 loss 3.704e-03 | train   7.409% | test   7.659% | mass  41.97% | drift  43.25% | data 8.21e-04 | w_eik 1.487e-02 | 1.1m
   750 loss 2.652e-03 | train   6.386% | test   6.768% | mass  42.16% | drift  43.45% | data 5.62e-04 | w_eik 1.285e-02 | 1.7m
  1000 loss 2.042e-03 | train   5.400% | test   5.799% | mass  35.92% | drift  36.97% | data 3.80e-04 | w_eik 1.100e-02 | 2.3m
  1250 loss 1.903e-03 | train   4.793% | test   5.132% | mass  33.27% | drift  34.24% | data 2.91e-04 | w_eik 9.537e-03 | 2.8m
  1500 loss 1.607e-03 | train   4.622% | test   4.871% | mass  35.82% | drift  36.85% | data 3.03e-04 | w_eik 8.371e-03 | 3.4m
  1750 loss 1.315e-03 | train   4.438%

In [ ]:
!python train_operator.py --benchmark rv --loss strong --saw --saw_init zero --n_labelled 8 --steps 20000 --n_train 16 --n_test 100 --seed 44

device=cuda  grid=64x64x32  loss=strong
benchmark: RV  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 8/16 instances  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 4.702e-03 | train   8.979% | test   8.895% | mass  46.21% | drift  47.69% | data 5.30e-04 | w_eik 1.702e-02 | 0.6m
   500 loss 3.345e-03 | train   7.784% | test   7.708% | mass  40.33% | drift  41.61% | data 5.68e-04 | w_eik 1.548e-02 | 1.1m
   750 loss 2.764e-03 | train   6.738% | test   6.687% | mass  33.66% | drift  34.71% | data 4.64e-04 | w_eik 1.372e-02 | 1.7m
  1000 loss 2.632e-03 | train   5.941% | test   5.880% | mass  34.34% | drift  35.44% | data 5.35e-04 | w_eik 1.194e-02 | 2.2m
  1250 loss 2.478e-03 | train   5.083% | test   5.075% | mass  28.09% | drift  28.99% | data 5.35e-04 | w_eik 1.047e-02 | 2.8m
  1500 loss 2.079e-03 | train   4.668% | test   4.681% | mass  30.52% | drift  31.52% | data 4.35e-04 | w_eik 9.056e-03 | 3.4m
  1750 loss 1.439e-03 | train   4.408%

In [ ]:
!python collect_results.py

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
hybrid-8    fixed   16    64^2x32  20000 42   0.858%   0.921%    2.63%   44.9
hybrid-8    fixed   16    64^2x32  20000 43   0.992%   1.077%    3.34%   44.8
hybrid-8    fixed   16    64^2x32  20000 44   0.943%   0.960%    2.23%   44.9
data-free   fixed   16    64^2x32  20000 42   1.503%   1.543%    2.67%   45.0
data-free   fixed   16    64^2x32  20000 43   1.595%   1.624%    3.94%   44.9
data-free   fixed   16    64^2x32  20000 44   1.658%   1.676%    4.11%   44.9
supervised  fixed   16    64^2x32  20000 42   0.256%   0.368%    1.14%   42.9
supervised  fixed   16    64^2x32  20000 43   0.262%   0.405%    1.22%   43.9
supervised  fixed   16    64^2x32  20000 44   0.284%   0.335%    1.03%   44.0

arm         v        n  steps seeds  test mean     std  mass mean     std
-------------------------------------------------------------------

---
## Results

In [ ]:
!python collect_results.py --sort test

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
supervised  fixed   16    64^2x32  20000 44   0.284%   0.335%    1.03%   44.0
supervised  fixed   16    64^2x32  20000 42   0.256%   0.368%    1.14%   42.9
supervised  fixed   16    64^2x32  20000 43   0.262%   0.405%    1.22%   43.9
hybrid-8    fixed   16    64^2x32  20000 42   0.858%   0.921%    2.63%   44.9
hybrid-8    fixed   16    64^2x32  20000 44   0.943%   0.960%    2.23%   44.9
hybrid-8    fixed   16    64^2x32  20000 43   0.992%   1.077%    3.34%   44.8
data-free   fixed   16    64^2x32  20000 42   1.503%   1.543%    2.67%   45.0
data-free   fixed   16    64^2x32  20000 43   1.595%   1.624%    3.94%   44.9
data-free   fixed   16    64^2x32  20000 44   1.658%   1.676%    4.11%   44.9

arm         v        n  steps seeds  test mean     std  mass mean     std
-------------------------------------------------------------------

In [ ]:
!python eval_checkpoint.py rv_strong_saw_saw0_n16_20k_64x32_s42 --n_test 100


rv_strong_saw_saw0_n16_20k_64x32_s42   loss=strong  IC=hard  n_train=16  grid=64^2 x 32  bench=RV  v=fixed  n_test=100
set         rel L2     abs L2   mass(ref)  mass(pi R^2)     drift
-----------------------------------------------------------------
train       1.503%  5.488e-03       2.29%         2.25%     2.21%
test        1.543%  5.629e-03       2.67%         2.54%     2.80%
(exact)     0.000%  0.000e+00       0.00%         0.88%     1.14%


In [ ]:
!python eval_checkpoint.py rv_supervised_n16_20k_64x32_s42 --n_test 100


rv_supervised_n16_20k_64x32_s42   loss=supervised  IC=hard  n_train=16  grid=64^2 x 32  bench=RV  v=fixed  n_test=100
set         rel L2     abs L2   mass(ref)  mass(pi R^2)     drift
-----------------------------------------------------------------
train       0.256%  9.323e-04       0.74%         1.07%     1.42%
test        0.368%  1.347e-03       1.14%         1.40%     1.52%
(exact)     0.000%  0.000e+00       0.00%         0.88%     1.14%


In [ ]:
!python eval_checkpoint.py rv_strong_saw_lab8_saw0_n16_20k_64x32_s42 --n_test 100


rv_strong_saw_lab8_saw0_n16_20k_64x32_s42   loss=strong  IC=hard  n_train=16  grid=64^2 x 32  bench=RV  v=fixed  n_test=100
set         rel L2     abs L2   mass(ref)  mass(pi R^2)     drift
-----------------------------------------------------------------
train       0.858%  3.127e-03       2.26%         2.21%     2.24%
test        0.921%  3.347e-03       2.63%         2.54%     2.69%
(exact)     0.000%  0.000e+00       0.00%         0.88%     1.14%


### Figures

Contours are what matter: a relative L2 number cannot distinguish a slightly
displaced interface from a smeared one.

In [ ]:
!python visualize.py rv_strong_saw_saw0_n16_20k_64x32_s42 rv_supervised_n16_20k_64x32_s42 rv_strong_saw_lab8_saw0_n16_20k_64x32_s42 --n_test 100

plotting instance 44: p=[0.451 0.759 0.117]
  physics only           mean 1.543%  this instance 1.516%
  supervised             mean 0.368%  this instance 0.235%
  hybrid (8 labels)      mean 0.921%  this instance 0.820%

wrote interfaces_rv_strong_saw_saw0_n16_20k_64x32_s42.png, fields_rv_strong_saw_saw0_n16_20k_64x32_s42.png, spread_rv_strong_saw_saw0_n16_20k_64x32_s42.png


In [ ]:
!python visualize.py rv_strong_saw_saw0_n16_20k_64x32_s42 rv_supervised_n16_20k_64x32_s42 rv_strong_saw_lab8_saw0_n16_20k_64x32_s42 --n_test 100 --instance 63

plotting instance 63: p=[0.513 0.7   0.145]
  physics only           mean 1.543%  this instance 1.451%
  supervised             mean 0.368%  this instance 0.362%
  hybrid (8 labels)      mean 0.921%  this instance 0.914%

wrote interfaces_rv_strong_saw_saw0_n16_20k_64x32_s42.png, fields_rv_strong_saw_saw0_n16_20k_64x32_s42.png, spread_rv_strong_saw_saw0_n16_20k_64x32_s42.png
